# B — Exploratory spatial data analysis

We test whether mining intensity and mining-related conflict are spatially clustered using Global Moran's I and its local decomposition (LISA). Analysis is restricted to the mining-relevant subset of 28,703 hexes — those within approximately 35 km (24 H3 rings at res 7) of any USGS concession. Running Moran's I on the full 383,693-hex grid is uninformative because 99.7% of cells have zero mining, so the statistic is dominated by zero-to-zero comparisons.

In [17]:
from pathlib import Path

import contextily as ctx
import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from esda.moran import Moran, Moran_Local
from libpysal.weights import W
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

In [18]:
PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "submission" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR   = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

hex_features = gpd.read_file(PROCESSED_DIR / "hex_features.gpkg")
print(f"{len(hex_features):,} hexes, {hex_features.shape[1]} columns")
print(f"Non-zero Liu: {(hex_features['liu_ha_erased'] > 0).sum():,}")
print(f"Non-zero incidents: {(hex_features['n_inc_mining_total'] > 0).sum():,}")

383,693 hexes, 24 columns
Non-zero Liu: 1,073
Non-zero incidents: 140


## Mining-relevant subset and spatial weights

The spatial weights matrix is built directly from the H3 adjacency structure: each cell's six `grid_disk(cell, 1)` neighbours, filtered to cells present in the subset and row-standardised. This is equivalent to queen contiguity on a hexagonal grid and avoids the computational cost of polygon intersection on 28k cells.

In [19]:
RING_RADIUS = 24
concession_hex = set(
    hex_features.loc[
        (hex_features["asm_share"] > 0) | (hex_features["lsm_share"] > 0), "hex_id"
    ]
)
candidate = set()
for hid in concession_hex:
    candidate |= set(h3.grid_disk(hid, RING_RADIUS))
sample = hex_features[hex_features["hex_id"].isin(candidate)].copy()
print(f"Mining-relevant subset: {len(sample):,} hexes")

sample_set = set(sample["hex_id"])
neighbors = {
    hid: [n for n in (set(h3.grid_disk(hid, 1)) - {hid}) if n in sample_set]
    for hid in sample["hex_id"]
}
w = W(neighbors, {k: [1.0] * len(v) for k, v in neighbors.items()}, silence_warnings=True)
w.transform = "r"
# Align row order to w.id_order so y-vectors are consistent with the weights
sample = sample.set_index("hex_id").loc[w.id_order].reset_index()
print(f"Mean neighbours per cell: {w.mean_neighbors:.2f}")

Mining-relevant subset: 28,703 hexes
Mean neighbours per cell: 5.93


## Global Moran's I

Both variables are log1p-transformed before testing because their distributions are strongly right-skewed. Significance is assessed via 999 random permutations.

In [20]:
sample["liu_log1p"] = np.log1p(sample["liu_ha_erased"])
sample["inc_log1p"] = np.log1p(sample["n_inc_mining_total"])


def global_moran(y, w, label, permutations=999):
    mi = Moran(y, w, permutations=permutations)
    print(f"{label:45s}  I = {mi.I:+.3f}   p = {mi.p_sim:.3f}")
    return mi


mi_mine = global_moran(sample["liu_log1p"].values, w, "log1p(Liu Ha_erased) — mining intensity")
mi_inc  = global_moran(sample["inc_log1p"].values,  w, "log1p(n_inc_mining_total) — incidents")

log1p(Liu Ha_erased) — mining intensity        I = +0.592   p = 0.001
log1p(n_inc_mining_total) — incidents          I = +0.031   p = 0.001


## LISA — Local Moran's I

Local significance is assessed via Benjamini–Hochberg FDR correction (q = 0.05) applied to analytical z-score p-values, rather than the 999-permutation pseudo-p. The permutation granularity (floor at 1/1000) is too coarse for a ~28,000-test correction.

In [21]:
FDR_Q = 0.05
QUAD_LABELS = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}


def run_lisa(y, w, df, col_prefix):
    """Run LISA, apply BH-FDR correction, write class column to df."""
    local = Moran_Local(y, w, permutations=999, seed=42)
    p_anal = 2 * (1 - norm.cdf(np.abs(local.z_sim)))
    reject, _, _, _ = multipletests(p_anal, alpha=FDR_Q, method="fdr_bh")
    df[f"{col_prefix}_class"] = np.where(
        reject, pd.Series(local.q).map(QUAD_LABELS), "ns"
    )
    counts = df[f"{col_prefix}_class"].value_counts()
    print(f"{col_prefix}: {dict(counts)}")
    return local


lisa_mine = run_lisa(sample["liu_log1p"].values,  w, sample, "lisa_mine")
lisa_inc  = run_lisa(sample["inc_log1p"].values,   w, sample, "lisa_inc")

lisa_mine: {'ns': np.int64(27729), 'HH': np.int64(561), 'LH': np.int64(413)}
lisa_inc: {'ns': np.int64(28370), 'LH': np.int64(324), 'HH': np.int64(9)}


## Figure 1 — LISA cluster maps

Mining intensity (left) and incident counts (right), saved as separate PDFs without titles. Red = HH hot-spots, blue = LL cold-spots. Grey hexes are in the subset but not significant after FDR correction.

In [ ]:
LISA_COLORS = {
    "HH": "#d62728",
    "LL": "#1f77b4",
    "HL": "#ff9896",
    "LH": "#9ecae1",
    "ns": "#eeeeee",
}

CITIES = {
    "Lubumbashi": (27.47, -11.66),
    "Kolwezi":    (25.47, -10.72),
    "Likasi":     (26.73, -10.99),
    "Ndola":      (28.63, -12.97),
    "Kitwe":      (28.21, -12.82),
    "Chingola":   (27.87, -12.53),
    "Solwezi":    (26.39, -12.17),
    "Mufulira":   (28.23, -12.55),
    "Luena":      (22.46, -11.78),
}

COUNTRY_LABELS = {
    "DRC":    (24.5, -9.2),
    "Zambia": (28.5, -14.2),
    "Angola": (15.5, -12.0),
}

cities_gdf = gpd.GeoDataFrame(
    {"name": list(CITIES.keys())},
    geometry=gpd.points_from_xy(
        [lon for lon, lat in CITIES.values()],
        [lat for lon, lat in CITIES.values()],
    ),
    crs="EPSG:4326",
).to_crs(3857)

country_pts = {
    name: gpd.GeoDataFrame(
        geometry=gpd.points_from_xy([lon], [lat]), crs="EPSG:4326"
    ).to_crs(3857).geometry[0]
    for name, (lon, lat) in COUNTRY_LABELS.items()
}


def save_lisa_map(gdf, col, fname):
    gdf_w = gdf.to_crs(3857)
    fig, ax = plt.subplots(figsize=(10, 8))
    gdf_w[gdf_w[col] == "ns"].plot(ax=ax, color="#aaaaaa", edgecolor="none", alpha=0.15)
    for cls in ["LL", "LH", "HL", "HH"]:
        sub = gdf_w[gdf_w[col] == cls]
        if len(sub):
            sub.plot(ax=ax, color=LISA_COLORS[cls], edgecolor="none",
                     alpha=0.85, label=f"{cls} (n={len(sub):,})")
    minx, miny, maxx, maxy = gdf_w.total_bounds
    pad = 50_000
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)
    ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.CartoDB.Positron, zoom=10)

    # Cities — only those inside the current extent
    xl, xr = ax.get_xlim()
    yb, yt = ax.get_ylim()
    visible_cities = cities_gdf[
        cities_gdf.geometry.x.between(xl, xr) &
        cities_gdf.geometry.y.between(yb, yt)
    ]
    visible_cities.plot(ax=ax, color="black", markersize=5, marker="o", zorder=6)
    for _, row in visible_cities.iterrows():
        ax.annotate(
            row["name"],
            xy=(row.geometry.x, row.geometry.y),
            xytext=(5, 4), textcoords="offset points",
            fontsize=7.5, fontweight="bold", color="#111111", zorder=7,
        )

    # Country labels — only if centroid falls inside the extent
    for name, pt in country_pts.items():
        if xl < pt.x < xr and yb < pt.y < yt:
            ax.text(
                pt.x, pt.y, name,
                fontsize=13, color="#555555", fontweight="bold",
                ha="center", va="center", fontstyle="italic",
                alpha=0.75, zorder=5,
            )

    ax.legend(loc="lower left", fontsize=8, title="LISA quadrant (FDR-significant)")
    ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / fname, format="pdf", bbox_inches="tight", dpi=300)
    plt.show()
    print(f"Saved {fname}")


save_lisa_map(sample, "lisa_mine_class", "fig1a_lisa_mining.pdf")
save_lisa_map(sample, "lisa_inc_class",  "fig1b_lisa_incidents.pdf")

In [23]:
# Summary: where are the mining hot-spots?
hh = sample[sample["lisa_mine_class"] == "HH"]
print("HH (mining hot-spot) cells by country:")
print(hh["country"].value_counts(dropna=False))

print("\nHH (incident hot-spot) cells by country:")
hh_inc = sample[sample["lisa_inc_class"] == "HH"]
print(hh_inc["country"].value_counts(dropna=False))

HH (mining hot-spot) cells by country:
country
Dem. Rep. Congo    293
Zambia             268
Name: count, dtype: int64

HH (incident hot-spot) cells by country:
country
Zambia             5
Dem. Rep. Congo    4
Name: count, dtype: int64


In [24]:
# Merge LISA classes back to hex_features and save
lisa_merge = sample[["hex_id", "lisa_mine_class", "lisa_inc_class"]]
hex_features = hex_features.drop(
    columns=[c for c in ["lisa_mine_class", "lisa_inc_class"] if c in hex_features.columns]
).merge(lisa_merge, on="hex_id", how="left")
for c in ["lisa_mine_class", "lisa_inc_class"]:
    hex_features[c] = hex_features[c].fillna("not_in_sample")

out_path = PROCESSED_DIR / "hex_features.gpkg"
out_path.unlink(missing_ok=True)
hex_features.to_file(out_path, driver="GPKG")
print(f"Wrote {out_path} ({len(hex_features):,} hexes, {hex_features.shape[1]} cols)")

Wrote /Users/inge/Desktop/ITU/GeoDS/gds/data/processed/hex_features.gpkg (383,693 hexes, 24 cols)
